# 4장 1강: 데이터 불균형과 평가지표 왜곡

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 데이터 불균형이 분류 모델 평가에 어떤 영향을 주는지 확인합니다.

- `Class` 비율을 확인하여 불균형 정도를 파악합니다.
- Accuracy만 보면 생길 수 있는 착시를 확인합니다.
- Precision, Recall, F1-score, PR-AUC를 함께 확인합니다.
- `class_weight`, Random Undersampling, SMOTE를 적용하여 성능 변화를 비교합니다.
- threshold 변화에 따른 Precision / Recall 변화를 확인합니다.

> 이 실습에서는 4장 1강 교안에 나온 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습에서는 모델 입력으로 `V1`부터 `V28`까지 사용하고, `Class`를 예측 대상으로 사용합니다.

## 실습 준비

1. 데이터 크기를 확인합니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 X, `Class`를 y로 분리합니다.
4. Train / Test 데이터를 분리합니다.
5. 리샘플링은 **Train 데이터에만** 적용합니다.

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

data = pd.read_csv("creditcard.csv")

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(data["Class"].value_counts().to_frame("count"))

print("\n[Class 비율]")
display(data["Class"].value_counts(normalize=True).to_frame("ratio"))

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain 크기:", X_train.shape)
print("Test 크기:", X_test.shape)

데이터 크기: (284807, 31)

[Class 개수]


,count
Class,
0,284315
1,492



[Class 비율]


,ratio
Class,
0,0.998273
1,0.001727



Train 크기: (227845, 28)
Test 크기: (56962, 28)


---

# 필수 1. Accuracy의 함정과 불균형 지표 확인

## 배경

Credit Card Fraud 데이터는 정상 거래가 매우 많고 사기 거래가 매우 적은 불균형 데이터입니다.

이런 상황에서는 Accuracy만 보면 모델이 매우 좋아 보일 수 있습니다.

## 문제 1. 모든 거래를 정상이라고 예측해보세요.

### 요구사항

1. Test 데이터의 모든 거래를 `0`으로 예측합니다.
2. Accuracy를 계산합니다.
3. Recall을 계산합니다.
4. Accuracy와 Recall 결과를 비교합니다.

### 확인 포인트

- 모든 거래를 정상이라고 예측해도 Accuracy가 매우 높게 나오는가?
- 사기 거래를 하나도 잡지 못하면 Recall은 어떻게 되는가?

# TODO
# 모든 거래를 정상(0)으로 예측하고 Accuracy / Recall을 계산하세요.
y_pred_zero = np.zeros(len(y_test))

print(f'Accuracy: {accuracy_score(y_test, y_pred_zero)}')
print(f'Recall: {recall_score(y_test, y_pred_zero)}')

### Q1. Accuracy가 높더라도 이 모델을 좋은 사기 탐지 모델이라고 보기 어려운 이유는 무엇인가요?

**답변:**  Test 데이터에 사기 거래가 98건밖에 없어서(약 0.17%), 전부 정상으로 찍어도 Accuracy가 99.8%나 나왔다. Recall은 0이라서 사기 거래를 하나도 못 잡았다. 사기를 찾는 게 목적인데 하나도 못 찾으니 Accuracy가 높아도 좋은 모델이라고 할 수 없다

## 문제 2. 기본 Logistic Regression의 여러 지표를 확인하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 학습합니다.
2. Test 데이터를 예측합니다.
3. Accuracy를 계산합니다.
4. Precision을 계산합니다.
5. Recall을 계산합니다.
6. F1-score를 계산합니다.
7. `predict_proba()`의 양성 클래스 확률로 PR-AUC를 계산합니다.
8. 혼동행렬을 확인합니다.

### 확인 포인트

- Accuracy 하나만 보는 것과 여러 지표를 함께 보는 것의 차이를 이해했는가?
- Precision과 Recall이 각각 어떤 오류와 연결되는지 설명할 수 있는가?
- PR-AUC가 불균형 데이터에서 왜 중요한지 이해했는가?

In [6]:
# TODO
# 기본 Logistic Regression의 Accuracy / Precision / Recall / F1 / PR-AUC / 혼동행렬을 확인하세요.
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(f'Recall: {recall_score(y_test, y_pred)}')
print(f'Precision: {precision_score(y_test, y_pred)}')
print(f'F1 - Score: {f1_score(y_test, y_pred)}')
print(f'PR - AUC: {average_precision_score(y_test, y_prob)}')

print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9991573329588147
Recall: 0.6428571428571429
Precision: 0.8289473684210527
F1 - Score: 0.7241379310344828
PR - AUC: 0.7435600673915171
[[56851    13]
 [   35    63]]


### Q2. 사기 거래를 놓치는 FN의 비용이 크다면 Precision과 Recall 중 무엇을 더 중요하게 봐야 하나요?

**답변:**  Recall을 더 중요하게 봐야 한다. Recall은 실제 사기 중에서 모델이 잡아낸 비율이라서 FN이 많을수록 낮아진다. 기본 모델은 Accuracy가 0.9992인데 Recall은 0.6429라서, 사기 98건 중 35건을 놓쳤다.

---

# 필수 2. 불균형 처리 기법 비교

## 배경

교안에서는 불균형 처리 방법으로 다음 세 가지를 다룹니다.

- `class_weight`
- Random Undersampling
- SMOTE

이번 문제에서는 세 방법을 Train 데이터에만 적용한 뒤 같은 Test 데이터에서 성능을 비교합니다.

## 문제 1. class_weight를 적용하세요.

### 요구사항

1. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
2. Train 데이터로 학습합니다.
3. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스에 더 큰 가중치를 주면 Recall이 어떻게 변하는가?
- Precision과 Recall이 동시에 항상 좋아지는 것은 아니라는 점을 확인했는가?

In [7]:
# TODO
# class_weight="balanced" 모델을 학습하고 지표를 확인하세요.
cw_model = LogisticRegression(class_weight='balanced', max_iter=1000)
cw_model.fit(X_train, y_train)

y_pred_cw = cw_model.predict(X_test)
y_proba_cw = cw_model.predict_proba(X_test)[:, 1]

print("Precision:", precision_score(y_test, y_pred_cw))
print("Recall:", recall_score(y_test, y_pred_cw))
print("F1-score:", f1_score(y_test, y_pred_cw))
print("PR-AUC:", average_precision_score(y_test, y_proba_cw))
print(confusion_matrix(y_test, y_pred_cw))

Precision: 0.05787781350482315
Recall: 0.9183673469387755
F1-score: 0.1088929219600726
PR-AUC: 0.7092143497034353
[[55399  1465]
 [    8    90]]


- Recall은 0.64에서 0.92로 올랐지만 Precision은 0.83에서 0.06으로 크게 떨어졌다. 정상거래를 사기로 잘못 본 FP가 13건에서 1465건으로 늘었다.

## 문제 2. Random Undersampling을 적용하세요.

### 요구사항

1. `RandomUnderSampler(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 다수 클래스 샘플이 줄어드는가?
- Test 데이터에는 Undersampling을 적용하지 않았는가?

In [10]:
# TODO
# Random Undersampling을 Train 데이터에만 적용하세요.
rus_model = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus_model.fit_resample(X_train, y_train)
print(f'적용 전: {y_train.value_counts().to_dict()}')
print(f'적용 후: {y_train_rus.value_counts().to_dict()}')

rus = LogisticRegression(max_iter=1000)
rus.fit(X_train_rus, y_train_rus)

y_pred_rus = rus.predict(X_test)
y_prob_rus = rus.predict_proba(X_test)[:, 1]

print(f'Precision: {precision_score(y_test, y_pred_rus)}')
print(f'Recall: {recall_score(y_test, y_pred_rus)}')
print(f'F1 - Score: {f1_score(y_test, y_pred_rus)}')
print(f'PR - AUC: {average_precision_score(y_test, y_pred_rus)}')

적용 전: {0: 227451, 1: 394}
적용 후: {0: 394, 1: 394}
Precision: 0.03759398496240601
Recall: 0.9183673469387755
F1 - Score: 0.07223113964686999
PR - AUC: 0.03466553273764526


- 정상 거래가 227451건에서 394로 줄었다. Test는 건드리지 않고 원본 그대로 평가했다.

## 문제 3. SMOTE를 적용하세요.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스 샘플이 증가하는가?
- SMOTE가 Test 데이터에 적용되지 않았는가?
- Recall 개선과 Precision 감소가 함께 나타날 수 있음을 확인했는가?

In [14]:
# TODO
# SMOTE를 Train 데이터에만 적용하세요.
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f'적용 전: {y_train.value_counts().to_dict()}')
print(f'적용 후: {y_train_sm.value_counts().to_dict()}')

sm_model = LogisticRegression(max_iter=1000)
sm_model.fit(X_train_sm, y_train_sm)

y_pred_sm = sm_model.predict(X_test)
y_prob_sm = sm_model.predict_proba(X_test)[:, 1]

print("Precision:", precision_score(y_test, y_pred_sm))
print("Recall:", recall_score(y_test, y_pred_sm))
print("F1-score:", f1_score(y_test, y_pred_sm))
print("PR-AUC:", average_precision_score(y_test, y_prob_sm))

적용 전: {0: 227451, 1: 394}
적용 후: {0: 227451, 1: 227451}
Precision: 0.05555555555555555
Recall: 0.9183673469387755
F1-score: 0.10477299185098952
PR-AUC: 0.718834211008162


### Q3. 리샘플링을 Test 데이터에도 적용하면 안 되는 이유는 무엇인가요?

**답변:**  SMOTE를 적용하면 진짜가 아닌 합성 데이터로 평가하게 된다. 그래서 리샘플링은 Train에만 해야 한다. -> 데이터 누수가 발생한다.(문제점)

---

# 과제 1. 불균형 처리 전후 성능과 threshold 비교

## 배경

필수 실습에서는 기본 모델과 세 가지 불균형 처리 방법을 확인했습니다.

이번 과제에서는 **기본 Logistic Regression과 class_weight 모델**을 비교하고, class_weight 모델에서 threshold에 따른 Precision / Recall 변화를 확인합니다.

> 과제는 필수 실습에서 사용한 지표와 방법을 그대로 비교하는 수준입니다.

## 요구사항

1. 기본 Logistic Regression과 `class_weight="balanced"` 모델의 다음 지표를 하나의 DataFrame으로 정리합니다.
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - PR-AUC
2. 두 모델의 Recall과 Precision 차이를 설명합니다.
3. `class_weight="balanced"` 모델의 양성 확률을 사용합니다.
4. threshold를 `0.3`, `0.5`, `0.7`로 바꾸어 예측합니다.
5. 각 threshold에서 Precision과 Recall을 계산합니다.
6. threshold 변화에 따라 두 지표가 어떻게 달라지는지 설명합니다.
7. 사기 거래를 놓치는 FN의 비용이 크다고 가정할 때, threshold를 낮추는 방향이 어떤 의미인지 설명합니다.

8. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만 사용할 때의 문제를 설명합니다.
9. FN의 비용과 데이터 특성을 고려해 `class_weight`, Random Undersampling, SMOTE 중 우선 적용할 방법을 하나 선택하고 이유와 한계를 설명합니다. 추가 모델 구현은 요구하지 않습니다.
10. Q4~Q7에 모두 답합니다.

> 처리 기법 선택은 근거 있는 제안으로 작성합니다. Test 결과를 보고 처리 기법이나 threshold를 반복 조정하지 않으며, 실제 설정 선택에는 Train 내부 검증 또는 교차검증이 필요합니다.

### 확인 포인트

- 불균형 처리 전후를 Accuracy 하나가 아니라 여러 지표로 비교했는가?
- threshold를 낮추면 Recall이 높아지는 경향을 설명할 수 있는가?
- threshold를 높이면 Precision이 높아지는 경향을 설명할 수 있는가?
- 단일한 정답 threshold가 있는 것이 아니라 비용 구조에 따라 결정해야 한다는 점을 이해했는가?
- 클래스 비율 때문에 Accuracy가 사기 탐지 성능을 오해하게 할 수 있음을 설명했는가?
- FN 비용과 데이터 특성을 근거로 처리 기법의 선택 이유·한계를 제시했는가?


In [15]:
# TODO
# 기본 모델 / class_weight 모델 비교와 threshold 0.3, 0.5, 0.7 비교를 진행하세요.
result = pd.DataFrame({
    'Accuracy': [accuracy_score(y_test, y_pred), accuracy_score(y_test, y_pred_cw)],
    'Precision': [precision_score(y_test, y_pred), precision_score(y_test, y_pred_cw)],
    'Recall': [recall_score(y_test, y_pred), recall_score(y_test, y_pred_cw)],
    'F1 - Score': [f1_score(y_test, y_pred), f1_score(y_test, y_pred_cw)],
    'PR - AUC': [average_precision_score(y_test, y_pred), average_precision_score(y_test, y_pred_cw)]
})
display(result)

for i in [0.3, 0.5, 0.7]:
    pred_i = (y_proba_cw >= i).astype(int)
    print(f'threshold = {i} Precision: {precision_score(y_test, pred_i)} Recall: {recall_score(y_test, pred_i)}')
    print(confusion_matrix(y_test, pred_i))

,Accuracy,Precision,Recall,F1 - Score,PR - AUC
0,0.999157,0.828947,0.642857,0.724138,0.533509
1,0.974141,0.057878,0.918367,0.108893,0.053294


threshold = 0.3 Precision: 0.02586206896551724 Recall: 0.9183673469387755
[[53474  3390]
 [    8    90]]
threshold = 0.5 Precision: 0.05787781350482315 Recall: 0.9183673469387755
[[55399  1465]
 [    8    90]]
threshold = 0.7 Precision: 0.11458333333333333 Recall: 0.8979591836734694
[[56184   680]
 [   10    88]]


- class_weight를 쓰니까 Recall은 0.6429에서 0.9184로 올랐다. 놓친 사기(FN)는 35건에서 8건으로 줄었다. 대신 Precision은 0.8289에서 0.0579로 많이 떨어졌다. 사기를 더 많이 잡으려다 보니 정상 거래도 사기로 많이 판단했기 때문이다(FP 13건에서 1,465건). 둘이 같이 좋아지지는 않는다는 걸 확인했다.
- threshold를 낮출수록 Precision은 떨어졌다(0.7일 때 0.1146, 0.5일 때 0.0579, 0.3일 때 0.0259). Recall은 0.7에서 0.5로 낮췄을 때 0.8980에서 0.9184로 올랐지만, 0.3까지 낮췄을 때는 그대로였다. 나머지 8건은 확률이 너무 낮게 나와서 threshold를 낮춰도 안 잡히는 것 같다.
- threshold를 낮춘다는 건 조금만 의심스러워도 사기로 보겠다는 뜻이다. 사기를 덜 놓치는 대신 정상 거래를 사기로 잘못 판단하는 경우가 늘어나는 걸 감수하는 것이다.

### Q4. 사기 거래를 놓치는 비용이 매우 크다면 threshold를 낮추는 방향은 어떤 장점과 단점이 있나요?

**답변:**  장점은 사기를 덜 놓친다는 것이다. 0.7에서 0.5로 낮추니까 사기 2건을 더 잡았다. 단점은 FP가 많이 늘어난다는 것이다. 0.3으로 낮추면 FP가 3,390건이나 되어 고객 불편이나 검토 비용이 커진다. 그리고 0.5에서 0.3으로 갈 때처럼 Recall은 그대로인데 FP만 늘어나는 경우도 있어서, 무조건 낮추는 게 좋은 건 아니다.

### Q5. CTGAN / TVAE와 SMOTE의 가장 큰 차이를 한 문장으로 설명하세요.

**답변:**  SMOTE는 기존 소수 클래스 데이터와 가까운 이웃 사이에 점을 찍어서 새 데이터를 만들고, CTGAN과 TVAE는 생성 모델이 데이터 분포 전체를 학습한 다음 새로운 데이터를 생성한다.

### Q6. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만으로 사기 탐지 성능을 판단할 때의 문제를 설명하세요.

**답변:** 전체 데이터 중 정상 거래가 99.83%라서, 정상으로만 예측해도 거의 다 맞힌 걸로 계산되어 Accuracy가 높게 나온다. 이 지표만 보면 사기를 하나도 못 잡는 모델도 좋아 보인다는 문제가 있다. 또 이번 결과처럼 기본 모델(Accuracy 0.9992)이 class_weight 모델(Accuracy 0.9741)보다 좋아 보이지만, 사기를 잡은 비율은 class_weight가 훨씬 높았다. 그래서 Recall, Precision, F1, PR-AUC를 같이 봐야 한다.


### Q7. 사기 거래를 놓치는 FN의 비용과 데이터 특성을 고려해 class_weight·Random Undersampling·SMOTE 중 우선 적용할 방법을 하나 선택하고, 선택 이유와 한계를 설명하세요.

**답변:** class_weight를 먼저 적용하겠다.
- 근거: undersampling은 정상 거래를 227,451건에서 394건으로 줄여서 정보를 거의 다 버리게 된다. SMOTE는 사기 데이터 대부분이 합성 데이터가 되고 데이터 양도 엄청 커진다. class_weight는 데이터를 그대로 두고 가중치만 바꾸면 되니까 간단하다. 실제 결과에서도 세 방법의 Recall이 0.9184로 같았는데, FP는 class_weight가 1,465건으로 가장 적었다(SMOTE 1,530건, Undersampling 2,304건).
- 한계: Precision이 0.0579로 너무 낮아서, 사기라고 경고한 것 중 대부분이 정상 거래다. PR-AUC도 기본 모델보다 조금 낮아서 모델 자체가 좋아졌다고 보기는 어렵다. 그리고 Test 한 번으로 본 결과라서 실제로 정하려면 교차검증으로 다시 확인해야 한다.


---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 불균형 데이터에서 Accuracy 하나만 보면 위험한 이유는 무엇인가요?
2. Precision은 어떤 오류가 중요할 때 보는 지표인가요?
3. Recall은 어떤 오류가 중요할 때 보는 지표인가요?
4. class_weight, Undersampling, SMOTE는 각각 어떤 방식으로 불균형을 다루나요?
5. 리샘플링은 Train / Test 중 어디에만 적용해야 하나요?
6. threshold를 낮추면 일반적으로 Recall과 Precision은 어떻게 변하나요?

- 다수 클래스만 예측해도 Accuracy가 높게 나와서, 소수 클래스를 못 잡는 모델도 좋아 보이기 때문이다.
- 정상을 사기로 잘못 판단하는 FP가 중요할 때 본다.
- 실제 사기를 놓치는 FN이 중요할 때 본다.
- class_weight는 소수 클래스에 가중치를 더 주고, Undersampling은 다수 클래스를 줄이고, SMOTE는 소수 클래스를 합성해서 늘린다.
- Train에만 적용해야 한다.
- Recall은 올라가고 Precision은 떨어진다.